# Remote Data Access: Query & Download Workflow

## Learning Objectives

- Configure authentication for remote catalogs
- Query STAC/OData catalogs (BIOMASS, Sentinel-1/2, NISAR, ASTER, VIIRS)
- Filter by spatial extent and time range
- Download products with progress tracking
- Use local caching for efficient re-use

## Theory: Catalog Protocols

| Protocol | Used By | Purpose |
|----------|---------|----------|
| **STAC** (SpatioTemporal Asset Catalog) | BIOMASS (ESA MAAP) | JSON-based metadata standard |
| **OData** | Sentinel-1/2 (CDSE) | Copernicus Data Space query API |
| **CMR** (Common Metadata Repository) | NISAR, ASTER, VIIRS (NASA Earthdata) | NASA's catalog protocol |

## Authentication Setup

**Option 1: Environment variables**
```bash
# Copernicus Data Space (Sentinel-1/2)
export CDSE_USERNAME=your_username
export CDSE_PASSWORD=your_password

# NASA Earthdata (NISAR, ASTER, VIIRS)
export EARTHDATA_USERNAME=your_username
export EARTHDATA_PASSWORD=your_password
```

**Option 2: Configuration file** (recommended for persistent credentials)  
Save to `~/.config/geoint/credentials.json`:
```json
{
  "cdse": {"username": "your_username", "password": "your_password"},
  "earthdata": {"username": "your_username", "password": "your_password"}
}
```

## Imports & Environment Validation

In [ ]:
"""GRDL Notebook Preamble — Environment validation and autoreload."""
try:
    get_ipython().run_line_magic('load_ext', 'autoreload')
    get_ipython().run_line_magic('autoreload', '2')
except (NameError, AttributeError):
    pass

import grdl
from packaging.version import Version

REQUIRED_VERSION = '0.6.1'
current_version = Version(grdl.__version__)
required_version = Version(REQUIRED_VERSION)

if current_version < required_version:
    raise RuntimeError(
        f"GRDL {REQUIRED_VERSION}+ required. Found {grdl.__version__}. "
        f"Update with: pip install --upgrade grdl"
    )

print(f"✓ GRDL {grdl.__version__} (>= {REQUIRED_VERSION})")

## Configuration

In [ ]:
from pathlib import Path

# Catalog selection
CATALOG_TYPE = 'biomass'  # Options: 'biomass', 'sentinel1', 'sentinel2', 'nisar', 'aster', 'viirs'

# Query parameters
START_DATE = '2023-01-01'
END_DATE = '2026-12-31'
MAX_RESULTS = 10
AOI_GEOJSON = None  # Path to GeoJSON AOI (optional spatial filter)

# Download configuration
DOWNLOAD_DIR = Path('/tmp/grdl_downloads')
ENABLE_DOWNLOAD = False  # Set True to actually download (requires valid credentials)

print(f"Catalog: {CATALOG_TYPE}")
print(f"Date range: {START_DATE} to {END_DATE}")
print(f"Max results: {MAX_RESULTS}")
print(f"Download directory: {DOWNLOAD_DIR}")
print(f"Download enabled: {ENABLE_DOWNLOAD}")

## Step 1: Initialize Catalog

GRDL provides catalog classes for each supported data source. Each handles protocol-specific authentication and query logic internally.

- SQLite catalog database will be stored (in `~/.config/geoint/catalogs/` by default)

Each catalog requires a `search_path` — the root directory where:- Local products will be discovered
- Downloaded products will be saved

In [ ]:
from grdl.IO.catalog import (
    BIOMASSCatalog,
    Sentinel1SLCCatalog,
    Sentinel2Catalog,
    NISARCatalog,
    ASTERCatalog,
    VIIRSCatalog,
)

# Catalog registry
catalog_map = {
    'biomass': BIOMASSCatalog,
    'sentinel1': Sentinel1SLCCatalog,
    'sentinel2': Sentinel2Catalog,
    'nisar': NISARCatalog,
    'aster': ASTERCatalog,
    'viirs': VIIRSCatalog,
}

if CATALOG_TYPE not in catalog_map:
    raise ValueError(f"Unknown catalog type: {CATALOG_TYPE}. Choose from {list(catalog_map.keys())}")

# Instantiate catalog with search_path (where products will be saved/discovered)
DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)
catalog = catalog_map[CATALOG_TYPE](search_path=DOWNLOAD_DIR)

print(f"✓ Catalog initialized: {type(catalog).__name__}")
print(f"  Search path: {DOWNLOAD_DIR}")

## Step 2: Query Products

Each catalog has its own query method depending on the remote API:

- **BIOMASS**: `query_esa()` (ESA MAAP STAC)All methods support temporal filtering and spatial bbox filtering.

- **Sentinel-1/2**: `query_cdse()` (Copernicus Data Space Ecosystem)
- **NISAR/ASTER/VIIRS**: `query_earthdata()` (NASA Earthdata CMR)

In [ ]:
# Query method mapping (catalog-specific APIs)
query_methods = {
    'biomass': 'query_esa',
    'sentinel1': 'query_cdse',
    'sentinel2': 'query_cdse',
    'nisar': 'query_earthdata',
    'aster': 'query_earthdata',
    'viirs': 'query_earthdata',
}

query_method_name = query_methods[CATALOG_TYPE]
query_method = getattr(catalog, query_method_name)

# Build query parameters
query_params = {
    'start_date': START_DATE,
    'end_date': END_DATE,
    'max_results': MAX_RESULTS,
}

# Add bbox for spatial filtering (optional)
if AOI_GEOJSON is not None:
    # For demonstration: bbox format is (min_lon, min_lat, max_lon, max_lat)
    # query_params['bbox'] = (115.5, -31.5, 116.8, -30.5)
    pass

# Execute query
print(f"Querying {CATALOG_TYPE} catalog using {query_method_name}()...")
results = query_method(**query_params)

print(f"\n✓ Found {len(results)} product(s)")

# Display results
if len(results) == 0:
    print("\n⚠ No products found. Try expanding date range or removing spatial filter.")
else:
    print("\nProduct listing:")
    print("=" * 80)
    for i, product in enumerate(results):
        product_id = product.get('id', product.get('title', 'Unknown'))
        date = product.get('date', 'N/A')
        size = product.get('size', 'N/A')
        
        print(f"\n{i+1}. {product_id}")
        print(f"   Date: {date}")
        print(f"   Size: {size}")
        
        if 'footprint' in product:
            footprint_preview = str(product['footprint'])[:100]
            print(f"   Footprint: {footprint_preview}...")

## Step 3: Download Products

All catalogs use `download_product(product_id)` to fetch products. Products are saved to the catalog's `search_path` configured in Step 1.

GRDL handles:
- Authentication token negotiation
- Progress tracking

- Local caching (skip re-download if already present)- Automatic extraction (for .zip/.tar archives)

In [ ]:
if not ENABLE_DOWNLOAD:
    print("\n⚠ Download disabled (ENABLE_DOWNLOAD=False)")
    print("  Set ENABLE_DOWNLOAD=True and configure credentials to download products.")
    print("  See authentication setup in the first cell for details.")
elif len(results) == 0:
    print("\n⚠ No products available for download")
else:
    # Download first product as demonstration
    product = results[0]
    product_id = product.get('id', product.get('title', 'Unknown'))
    
    print(f"\nDownloading product: {product_id}")
    print(f"  Date: {product.get('date', 'N/A')}")
    print(f"  Size: {product.get('size', 'N/A')}")
    print(f"  Destination: {DOWNLOAD_DIR}")
    print("\n" + "=" * 80)
    
    try:
        local_path = catalog.download_product(product_id)
        print(f"\n✓ Downloaded to: {local_path}")
        print(f"  File size: {Path(local_path).stat().st_size / (1024**3):.2f} GB")
    except Exception as e:
        print(f"\n✗ Download failed: {e}")
        print("  Verify credentials are configured correctly.")

## Summary

**GRDL patterns demonstrated**:
- ✅ `BIOMASSCatalog(search_path)`, `Sentinel1SLCCatalog(search_path)`, etc. — format-specific catalog clients
- ✅ Catalog-specific query methods:
  - `query_esa()` for BIOMASS (ESA MAAP STAC)
  - `query_cdse()` for Sentinel-1/2 (Copernicus Data Space)
  - `query_earthdata()` for NISAR/ASTER/VIIRS (NASA Earthdata CMR)
- ✅ `catalog.download_product(product_id)` — authenticated download with progress tracking
- ✅ Credential management via environment variables or `~/.config/geoint/credentials.json`

**Next steps**:
- Open downloaded products with format-specific readers (e.g., `grdl.IO.open_reader()`)
- See `IO/view_sicd.ipynb` for SAR product exploration
- See `ortho/ortho_sicd.ipynb` for orthorectification workflows